In [ ]:
/**
 * Design a URL Shortener:
 * - Ab hum next System Design problem start kar rahe hain:
 *   "URL Shortener kaise design karenge?"
 * - URL Shortener ka simple goal hai ek long URL ko short aur easy-to-share URL mein
 *   convert karna.
 *
 * Example:
 * - Long URL: https://www.example.com/products/category/electronics/
 *             smartphones/iphone/iphone-17-pro?color=black
 *                          |
 *                          v
 *                     URL Shortener
 *                          |
 *                          v
 * - Short URL: https://short.ly/aB3xK9
 *
 *
 * Jab user short URL open karega:
 * - https://short.ly/aB3xK9
 *             |
 *             v
 *       URL Shortener
 *             |
 *             v
 *   Find Original URL
 *             |
 *             v
 *   Redirect to Long URL
*/

/**
 * Simple words mein:
 * - URL Shortener ke mainly do kaam hain:
 *   1. Long URL ko short URL mein convert karna.
 *   2. Short URL ko original long URL par redirect karna.
 *
 * Basic Flow:
 * > Long URL
 * > Create Short URL
 * > Short URL
 * > User Opens Short URL
 * > Find Original URL
 * > HTTP Redirect
 * > Original Website
 *
 * Ab system design mein hum step-by-step decide karenge:
 * -> Requirements kya hain?
 * -> Kitne users/requests handle karne hain?
 * -> Short URL kaise generate hoga?
 * -> URL mapping kahan store hogi?
 * -> Read aur write path kaise work karega?
 * -> Database kaunsa use karenge?
 * -> System ko kaise scale karenge?
 * -> Duplicate/custom URLs kaise handle karenge?
 * -> Expired URLs kaise handle karenge?
 * -> High availability aur low latency kaise achieve karenge?
*/

/**
 * Core Idea:
 *
 *         Long URL
 *            |
 *            v
 *       +------------+
 *       | URL Shortener|
 *       +------+-----+
 *              |
 *              v
 *        Short URL Code
 *              |
 *              v
 *       Store Mapping
 *
 *
 *         Short URL
 *            |
 *            v
 *       +--------------+
 *       | URL Shortener|
 *       +------+-------+
 *              |
 *              v
 *       Lookup Mapping
 *              |
 *              v
 *         Long URL
 *              |
 *              v
 *          Redirect
 *
 * - Yaani poore system ka basic idea hai:
 * - "Long URL aur short URL ke beech ek mapping maintain karo, aur short URL ko 
 *    request milne par us mapping ke through original URL par redirect karo."
 */

In [ ]:

/**
 * Functional and Non-Functional Requirements:
 * - URL Shortener design karne se pehle humein clear karna hoga ki system ko exactly
 *   kya kaam karna hai aur system ko kis level ki performance provide karni hai.
 *
 * 1. Functional Requirements:
 *    Functional Requirements batati hain: "System ko kya-kya kaam karna chahiye?"
 *
 *    a. URL Shortening
 *       - Sabse pehla kaam hai ek long URL ko short URL mein convert karna.
 *       - Input: Long URL -> URL Shortener -> Unique Short URL
 *       - Example:
 *         Input: https://www.example.com/products/electronics/smartphones/iphone
 *                              |
 *                              v
 *                        URL Shortener
 *                              |
 *                              v
 *         Output: https://short.ly/aB3xK9
 *       - Yaani: Long URL -> Unique Short URL
 *
 *       Important:
 *       - Generated short URL/code unique hona chahiye, taaki different long URLs 
 *         accidentally same short URL par map na ho jaayein.
 *
 *    b. URL Redirection
 *       - Second main kaam hai short URL ko original long URL par redirect karna.
 *              User
 *               |
 *               | Open Short URL
 *               v
 *           URL Shortener
 *               |
 *               v
 *           Find Original URL
 *               |
 *               v
 *           Long URL
 *               |
 *               v
 *            Redirect
 *
 *       - Example:
 *         - User opens: https://short.ly/aB3xK9
 *                                |
 *                                v
 *         - Lookup: aB3xK9 -> https://www.example.com/...
 *                                |
 *                  +-------------+
 *                  |
 *                  v
 *         - HTTP Redirect
 *                  |
 *                  v
 *         - Original Website
 *
 *       - Yaani: Short URL -> Original Long URL
 *
 *
 *
 * 2. Non-Functional Requirements:
 *    Non-Functional Requirements batati hain: "System ko ye kaam kitni efficiently
 *    aur reliably perform karna chahiye?"
 *
 *    a. High Availability
 *       - URL Shortener users ko mostly available rehna chahiye.
 *
 *             User
 *               |
 *               v
 *          Short URL
 *               |
 *               v
 *          URL Shortener
 *               |
 *               v
 *           Redirect
 *
 *       - Agar ek server fail ho jaaye, to poora system unavailable nahi hona chahiye.
 *
 *    b. Scalability
 *       - System ko large number of users aur traffic handle karne mein capable hona
 *         chahiye.
 *
 *            More Users
 *                 |
 *                 v
 *            More Requests
 *                 |
 *                 v
 *            More Traffic
 *                 |
 *                 v
 *          Scale Horizontally
 *                 |
 *                 v
 *          Multiple Servers
 *
 *      - Yaani traffic increase hone par hum system mein additional servers/resources
 *        add kar saken.
 *
 *    c. Low Latency for Redirection
 *       - URL Shortener mein redirection bahut fast honi chahiye.
 *  
 *           Short URL
 *               |
 *               v
 *            Lookup
 *               |
 *               v
 *            Redirect
 *
 *       - User ko unnecessary delay nahi milna chahiye.
 *       - Especially kyunki URL Shortener ka common operation redirection hai, isliye
 *         lookup path highly optimized hona chahiye.
 *
 *    d. Fault Tolerance
 *       - System ko server, network ya infrastructure failures ke baad bhi service 
 *         continue karne ki capability honi chahiye.
 *
 *            Server 1 -> FAILED
 *                X
 *   
 *            Server 2 -> HEALTHY
 *            Server 3 -> HEALTHY
 *   
 *                 |
 *                 v
 *   
 *            Service Continues
 *
 *       - Iske liye multiple servers, replication aur failure-handling mechanisms use
 *         kiye ja sakte hain.
*/

/**
 * Simple words mein:
 * 1. Functional Requirements:
 *    -> Long URL ko short URL mein convert karo.
 *    -> Short URL ko original URL par redirect karo.
 *
 * 2. Non-Functional Requirements:
 *    -> High Availability - System mostly available rahe.
 *    -> Scalability       - Large traffic handle kar sake.
 *    -> Low Latency       - Redirection fast ho.
 *    -> Fault Tolerance   - Failures ke baad bhi system continue kare.
 *
 * Core Idea:
 *
 *              URL Shortener
 *                    |
 *          +---------+---------+
 *          |                   |
 *          v                   v
 *      Functional         Non-Functional
 *      Requirements       Requirements
 *          |                   |
 *       +--+--+          +-----+-----+-----+
 *       |     |          |     |     |     |
 *       v     v          v     v     v     v
 *    Shorten Redirect    HA  Scale  Low   Fault
 *                                 Latency Tolerance
 *
 * - Yaani pehle hum define karte hain ki system ko kya karna hai, aur phir define karte
 *   hain ki system ko kis performance, availability aur reliability ke saath wo kaam 
 *   karna hai.
 */

In [ ]:

/**
 * Constraints:
 *
 *    Ab requirements ke baad hum system ki kuch important
 *    constraints define karte hain.
 *
 *    Constraints ka matlab hai:
 *
 *    "System ko kis scale aur kis fixed rules ke according design
 *     karna hai?"
 *
 *
 *    1. Traffic:
 *
 *    System ko approximately:
 *
 *         100 million URLs / day
 *
 *    generate karne honge.
 *
 *    Yaani har din around 10 crore new URLs create hongi.
 *
 *         100 Million URLs / Day
 *                  |
 *                  v
 *          Large Write Traffic
 *                  |
 *                  v
 *          Scalable System Required
 *
 *    Is scale ko handle karne ke liye system ko horizontally
 *    scale karne layak design karna hoga.
 *
 *
 *    2. Short URL Length:
 *
 *    Short URL ka code:
 *
 *         "As short as possible"
 *
 *    hona chahiye.
 *
 *    Example:
 *
 *         Long:
 *         https://www.example.com/products/12345
 *
 *         Short:
 *         https://short.ly/aB3xK9
 *
 *    Yaani humein aisa short code generate karna hai jo:
 *
 *         -> Unique ho
 *         -> Minimum length ka ho
 *         -> Easily usable ho
 *
 *    Short code chhota hone se:
 *
 *         Shorter Code
 *              |
 *              v
 *       Shorter URL
 *              |
 *              v
 *       Easy to Share
 *
 *
 *    3. Allowed Characters:
 *
 *    Short URL mein sirf ye characters allowed hain:
 *
 *         0-9
 *         a-z
 *         A-Z
 *
 *    In total:
 *
 *         10 digits
 *       + 26 lowercase letters
 *       + 26 uppercase letters
 *       -----------------------
 *         62 characters
 *
 *    Is character set ko:
 *
 *         Base62
 *
 *    kaha jaata hai.
 *
 *
 *    Base62:
 *
 *         0 1 2 3 ... 9
 *         a b c ... z
 *         A B C ... Z
 *
 *    Total = 62 characters
 *
 *
 *    Example short codes:
 *
 *         aB3xK9
 *         Z8p2Lm
 *         7xQa91
 *
 *    Base62 ka benefit ye hai ki relatively small number of
 *    characters mein bahut saare unique combinations represent
 *    kiye ja sakte hain.
 *
 *
 *    4. URL Mutability:
 *
 *    Ek baar short URL generate ho gaya, uske baad:
 *
 *         -> Short URL delete nahi kiya jaayega.
 *         -> Short URL update nahi kiya jaayega.
 *
 *    Example:
 *
 *         aB3xK9
 *            |
 *            v
 *      Original URL A
 *
 *    Ye mapping permanent rahegi.
 *
 *    Hum baad mein:
 *
 *         aB3xK9 -> Original URL B
 *
 *    nahi karenge.
 *
 *    Similarly short URL ko delete karke reuse bhi nahi karenge.
 *
 *
 *    Is constraint ka benefit:
 *
 *         Immutable Mapping
 *                |
 *                v
 *       Simpler Data Management
 *                |
 *                v
 *       No Update/Delete Complexity
 *                |
 *                v
 *       Easier Caching
 *
 *
 *    Complete Constraints:
 *
 *         +------------------------------------------------+
 *         | Constraint              | Requirement          |
 *         +-------------------------+----------------------+
 *         | Traffic                 | ~100M URLs/day       |
 *         | Short URL Length        | As short as possible |
 *         | Allowed Characters      | Base62               |
 *         | URL Mutability          | No update/delete    |
 *         +-------------------------+----------------------+
 *
 *
 *    Simple words mein:
 *
 *    Hamara URL Shortener large scale par operate karega:
 *
 *         ~100 million new URLs every day.
 *
 *    Short code:
 *
 *         -> As short as possible
 *         -> Base62 characters only
 *         -> Unique
 *
 *    Aur ek baar URL create hone ke baad uski mapping immutable
 *    rahegi — short URL ko update ya delete nahi kiya jaayega.
 *
 *
 *    Core Idea:
 *
 *         100M URLs / Day
 *                |
 *                v
 *         Scalable System
 *
 *         Short Code
 *                |
 *                v
 *            Base62
 *                |
 *                v
 *        As Short As Possible
 *
 *         Created URL
 *                |
 *                v
 *           Immutable
 *                |
 *                v
 *        No Update / Delete
 */


In [ ]:
/**
 * Back-of-the-Envelope Estimation:
 * - Ab hum quickly estimate karenge ki URL Shortener ko approximately kitna traffic 
 *   aur storage handle karna padega.
 * - Back-of-the-Envelope Estimation ka purpose exact number nikalna nahi hota.
 * - Iska purpose hai: "System ka scale roughly kitna hoga?"
 * - Isse humein database, cache, servers aur storage ke baare mein better design 
 *   decisions lene mein help milti hai.
 *
 * 1. Traffic Estimation:
 *    - Given: URLs generated per day = 100 million
 *    - Yaani: 100M URLs / day => 100M = 100,000,000
 *
 *    - Ab average write operations per second calculate karte hain.
 *    - Ek day mein:
 *      - 24 hours
 *      - 24 x 60 minutes
 *      - 24 x 60 x 60 seconds = 86,400 seconds
 *    - Therefore: Write Operations/sec
 *      = 100M / (24 x 3600)
 *      = 100,000,000 / 86,400
 *      ≈ 1,160 writes/sec
 *    - Yaani system ko average approximately: ~1,160 writes/sec handle karne honge.
 *
 *    - Ab maan lo Read : Write ratio: 10 : 1
 *    - Matlab approximately har 1 URL creation ke against 10 URL redirection/read 
 *      requests aa sakti hain.
 *    - Therefore: Read Operations/sec
 *      = 1,160 x 10
 *      = 11,600 reads/sec
 *    - So approximately:
 *      - Writes -> ~1,160/sec
 *      - Reads  -> ~11,600/sec
 *
 *    Traffic Flow:
 *
 *         Client Requests
 *               |
 *          +----+----+
 *          |         |
 *          v         v
 *       Write      Read
 *       ~1,160    ~11,600
 *        /sec        /sec
 *
 *    Important:
 *    - Ye average traffic hai.
 *    - Real production system mein traffic uniform nahi hota.
 *    - Peak hours mein requests average se kaafi zyada ho sakti hain.
 *    - Isliye system ko sirf average ~11,600 reads/sec ke liye nahi, balki peak 
 *      traffic ke liye bhi design karna hoga.
 *
 *
 * 2. Data Growth Estimation:
 *    - Ab estimate karte hain ki long term mein kitne URLs store honge.
 *    - Given:
 *      - Service Lifetime = 10 years
 *      - URLs generated/day = 100M
 *
 *    - Ek year mein approximately: 365 days
 *    - 10 years mein: 365 x 10 = 3,650 days
 * 
 *    - Total URLs: 100M x 365 x 10
 *      = 365 Billion URLs
 *      = 365B URLs
 *    - Yaani 10 years ke baad system ko approximately: ~365 Billion
 *
 *   - URL mappings store karni pad sakti hain.
 *   - Data Growth:
 *
 *         Day 1
 *            |
 *            v
 *         100M URLs
 *            |
 *            v
 *         1 Year
 *            |
 *            v
 *         ~36.5B URLs
 *            |
 *            v
 *         10 Years
 *            |
 *            v
 *         ~365B URLs
 *
 * 3. Storage Estimation:
 *    - Ab calculate karte hain ki itne URLs ko store karne ke liye approximately kitni
 *      storage required hogi.
 *    - Assume: Average URL size = 100 bytes
 *    - Is estimate mein hum simplified assumption use kar rahe hain ki ek stored URL 
 *      mapping ko approximately 100 bytes space chahiye.
 *
 *    - Total URLs: 365B
 *    - Storage: 365B x 100 bytes
 *      = 36.5 Trillion bytes
 *      ≈ 36.5 TB
 *    - Yaani approximately: ~36.5 TB raw storage required hogi.
 *
 *
 *    Important:
 *    - Ye sirf basic data ka estimate hai.
 *    - Real system mein actual storage isse zyada ho sakti hai because database 
 *      indexes, metadata, replication, compression overhead, logs aur other system 
 *      overhead bhi storage consume karenge.
 *
 *    Complete Estimation:
 *    +--------------------------------------+
 *    | Metric                  | Estimate   |
 *    +-------------------------+------------+
 *    | URLs generated/day      | 100M       |
 *    | Write traffic           | ~1,160/sec |
 *    | Read traffic            | ~11,600/sec|
 *    | Read : Write ratio      | 10 : 1     |
 *    | Service lifetime        | 10 years   |
 *    | Total URLs              | 365B       |
 *    | Avg URL size            | 100 bytes  |
 *    | Raw storage             | ~36.5 TB   |
 *    +-------------------------+------------+
*/

/**
 * Simple words mein:
 * - Hamara URL Shortener average: ~1,160 writes/sec => ~11,600 reads/sec
 *   handle karega.
 * - Agar system 10 years tak chale aur har din 100M URLs generate kare, to 
 *   approximately: 100M x 365 x 10 => 365B URLs store honge.
 * - Agar ek URL mapping average 100 bytes leti hai, to raw storage requirement 
 *   approximately: 365B x 100 bytes ≈ 36.5 TB
 *
 * Core Idea:
 *
 *  100M URLs/day
 *         |
 *         v
 *  ~1,160 writes/sec
 *         |
 *         v
 *   10 : 1 Read/Write
 *         |
 *         v
 *  11,600 reads/sec
 * 
 *     10 Years
 *        |
 *        v
 *   ~365B URLs
 *        |
 *        v
 *  ~36.5 TB Raw Data
 *
 * - Yaani Back-of-the-Envelope Estimation humein system ke approximate scale ka idea
 *   deta hai, jiske basis par hum architecture aur infrastructure choose kar sakte 
 *   hain.
 */

In [ ]:
/**
 * API Endpoints:
 * - Ab hum define karenge ki Client aur URL Shortener Service ek doosre ke saath kaise
 *   communicate karenge.
 * - Is communication ke liye hum RESTful APIs expose karenge.
 * - Hamare system mein mainly do important operations hain:
 *   1. Long URL ko short URL mein convert karna.
 *   2. Short URL ko original Long URL par redirect karna.
 *
 * Basic API Flow:
 *
 *   Client
 *      |
 *      +---- POST ----> Create Short URL
 *      |
 *      +---- GET -----> Redirect Short URL
 *
 * 1. Create Short URL
 *    - Client jab kisi long URL ko shorten karna chahta hai, to POST request bhejega.
 *    - Endpoint: POST /api/v1/data/shorten
 *    - Request:
 *      {
 *          "longUrl": "https://www.example.com/very/long/url"
 *      }
 *    - Yaani client system ko original long URL provide karega.
 *
 *         Client
 *            |
 *            | POST /api/v1/data/shorten
 *            | { longUrl: "https://..." }
 *            v
 *       URL Shortener
 *            |
 *            v
 *       Generate Unique ID
 *            |
 *            v
 *       Create Short URL
 *            |
 *            v
 *          Return
 *    - Response: shortURL
 *    - Example: https://short.ly/abc123
 *    - Purpose: "Provided long URL ke liye ek unique short URL generate karna."
 *
 *
 *    Complete Example:
 *    - Request: POST /api/v1/data/shorten
 *      {
 *          "longUrl":
 *          "https://www.example.com/very/long/url"
 *      }
 *                  |
 *                  v
 *    - URL Shortener Service
 *                  |
 *                  v
 *    - Generate: abc123
 *                  |
 *                  v
 *    - Response: https://short.ly/abc123
 *
 *
 * 2. Redirect Short URL
 *    - Jab user short URL open karta hai, to system ko us short code ka corresponding
 *      original long URL find karna hota hai.
 *    - Endpoint: GET /api/v1/shortUrl
 *    - Example : GET /api/v1/abc123
 *
 *         Client / Browser
 *                |
 *                | GET /api/v1/abc123
 *                v
 *          URL Shortener
 *                |
 *                v
 *          Lookup: abc123
 *                |
 *                v
 *          Original Long URL
 *                |
 *                v
 *          HTTP 301 / 302
 *                |
 *                v
 *       https://www.example.com/very/long/url
 *
 *
 *    - Server short URL ko database/cache mein lookup karega:
 *
 *         abc123
 *            |
 *            v
 *       Find Mapping
 *            |
 *            v
 *       Original URL
 *
 *
 *    - Example mapping:
 *
 *         abc123
 *            |
 *            v
 *         https://www.example.com/very/long/url
 *
 *
 *    - Uske baad server browser ko HTTP redirect response dega.
 *      - HTTP 301 OR HTTP 302
 *      - 301 Redirect: Permanent Redirect
 *      - 302 Redirect: Temporary Redirect
 *
 *    - Example: GET /api/v1/abc123
 *                |
 *                v
 *      Server finds:abc123 -> https://www.example.com/very/long/url
 *              |
 *              v
 *         HTTP 301/302 Redirect
 *              |
 *              v
 *         Original Website
 *
 *
 *  Complete API Summary:
 *  +-----------------------------------------------------+
 *  | Operation           | Method | Endpoint             |
 *  +---------------------+--------+----------------------+
 *  | Create Short URL    | POST   | /api/v1/data/shorten |
 *  | Redirect Short URL  | GET    | /api/v1/shortUrl     |
 *  +---------------------+--------+----------------------+
*/

/**
 * Simple words mein:
 * - URL Shortener ke paas do main APIs hain.
 * - Create API: POST /api/v1/data/shorten
 *
 *         Long URL
 *             |
 *             v
 *         Short URL
 *
 *
 * - Redirect API: GET /api/v1/abc123
 *
 *         Short URL
 *             |
 *             v
 *         Find Original URL
 *             |
 *             v
 *         HTTP 301/302
 *             |
 *             v
 *         Long URL
 *
 *
 * Core Idea:
 *
 *         +--------+
 *         | Client |
 *         +---+----+
 *             |
 *       +-----+-----+
 *       |           |
 *       v           v
 *     POST          GET
 *   /shorten     /abc123
 *       |           |
 *       v           v
 *   Generate     Lookup
 *   Short URL    Long URL
 *       |           |
 *       v           v
 *    Return      301/302
 *    Short URL   Redirect
 */

In [ ]:
/**
 * URL Redirecting:
 * - Ab hum samjhenge ki jab user short URL open karta hai, to system usse original 
 *   long URL par kaise redirect karta hai.
 * - Example:
 *
 *         Short URL
 *              |
 *              v
 *       https://tinyurl.com/abc123
 *              |
 *              v
 *         TinyURL Service
 *              |
 *              v
 *          Find Long URL
 *              |
 *              v
 *       Amazon Long URL
*/

In [ ]:
/**
 * Redirection Types:
 * 301 - Redirect (Permanent)
 * - Indicates URL is permanently moved
 * - Browser caches the redirect
 * - Future requests go directly to the long URL
 * - Advantage: Reduces server load
 * - Disadvantage: Limited analytics tracking
 * 
 * 302 - Redirect (Temporary)
 * - Indicates URL is temporarily moved
 * - Browser does not cache redirect
 * - Every request goes through the URL shortener service
 * - Advantage: Enables detailed analytics (clicks sources)
 * - Disadvantage: Higher server load
*/

/**
 * Complete Browser Flow:
 *
 *         User / Browser
 *                |
 *                | GET https://tinyurl.com/abc123
 *                v
 *         +-------------+
 *         |   TinyURL   |
 *         |   Service   |
 *         +------+------+
 *                |
 *                | Lookup abc123
 *                v
 *         +-------------+
 *         | Database /  | 
 *         |    Cache    |
 *         +------+------+
 *                |
 *                | Original URL
 *                v
 *         https://www.amazon.com/very/long/product/url
 *                |
 *                | HTTP 301 / 302
 *                v
 *         +-------------+
 *         |   Browser   |
 *         +------+------+
 *                |
 *                | Redirect Request
 *                v
 *         +-------------+
 *         |    Amazon   |
 *         +-------------+
 *
 * - Yaani TinyURL khud actual destination nahi hota.
 * - TinyURL sirf short code ko original long URL ke saath map karta hai.
 *
 *         abc123
 *            |
 *            v
 *         Amazon URL
 *
 * - Jab user abc123 open karta hai, TinyURL mapping lookup karta hai
 *   aur browser ko Amazon ke original URL par redirect kar deta hai.
 */

In [ ]:
/**
 * URL Shortening:
 * - Ab hum samjhenge ki Long URL ko Short URL mein convert karte time humein kaunse 
 *   important properties maintain karni hain.
 * - Basic idea simple hai: Long URL > Shortening Login > Short Code > Short URL
 *
 * 1. Deterministic Mapping
 *    - Same longURL ke liye shortening logic ko consistently same hashValue/short code
 *      generate karna chahiye.
 *    - Example:
 *      - Long URL: https://www.amazon.com/product/123
 *                                |
 *                                v
 *                          Shortening Function
 *                                |
 *                                v
 *                             qtj5opu
 *      - Agar same Long URL dobara process karein:
 *         https://www.amazon.com/product/123
 *                    |
 *                    v
 *              Shortening Function
 *                    |
 *                    v
 *                 qtj5opu
 *
 *       - Yaani:
 *                Same longURL
 *                     |
 *                     v
 *                Same hashValue
 *
 *       - Is property ko Deterministic Mapping kehte hain.
 *
 * 2. Unique Mapping
 *    - Har Long URL ko ek unique hashValue/short code ke saath map karna chahiye.
 *
 *      Long URL A
 *           |
 *           v
 *         qtj5opu
 *
 *      Long URL B
 *           |
 *           v
 *         x7k2Lm9
 *
 *   - Dono different Long URLs ko same short code nahi milna chahiye.
 *     - Long URL A -----> qtj5opu
 *     - Long URL B -----> x7k2Lm9
 *   - Yaani:
 *     - Different Long URLs
 *             |
 *             v
 *     - Different Short Codes
 *   - Agar do different URLs ko same short code mil gaya, to redirect ke time 
 *     system confuse ho jayega.
 *
 * 3. Reversible Lookup
 *    - Shortening ke baad system ko reverse lookup bhi karna aana chahiye.
 *    - Matlab:
 *
 *         Short Code
 *             |
 *             v
 *        Find Mapping
 *             |
 *             v
 *         Long URL
 *
 *     - Example:
 *
 *         qtj5opu
 *             |
 *             v
 *         https://www.amazon.com/product/123
 *
 *     - Isi mapping ki wajah se jab user short URL open karta hai,
 *       system original Long URL find karke redirect kar sakta hai.
*/

/**
 * Complete Mapping:
 *
 *         +-------------------------------+
 *         |          Long URL             |
 *         | https://www.amazon.com/...    |
 *         +---------------+---------------+
 *                         |
 *                         | Shortening Function
 *                         v
 *                    +---------+
 *                    |   fx    |
 *                    +----+----+
 *                         |
 *                         v
 *                    +---------+
 *                    | qtj5opu |
 *                    +----+----+
 *                         |
 *                         v
 *             https://tinyurl.com/qtj5opu
 *
 *
 * Reverse Lookup:
 *
 *         https://tinyurl.com/qtj5opu
 *                         |
 *                         v
 *                    Extract Code
 *                         |
 *                         v
 *                      qtj5opu
 *                         |
 *                         v
 *                  Database / Cache
 *                         |
 *                         v
 *         https://www.amazon.com/product/123
 *                         |
 *                         v
 *                      Redirect
 *
 *
 * Important Point:
 * - Yahan "Reversible Lookup" ka matlab ye nahi hai ki hash function mathematically 
 *   reverse kiya ja sakta hai.
 * - Instead, system ek mapping store karta hai: shortCode -> longURL
 * - Example:
 *
 *         qtj5opu
 *             |
 *             v
 *         https://www.amazon.com/product/123
 *
 * - Is stored mapping ki help se hum short code se original Long URL retrieve karte hain.
 *
 * 
 * Simple words mein:
 * - URL Shortening system ko 3 important things ensure karni hain:
 *   1. Deterministic: Same Long URL -> Same Short Code
 *   2. Unique: Different Long URLs -> Different Short Codes
 *   3. Reversible Lookup: Short Code -> Original Long URL
 *
 *
 * Core Idea:
 *
 *         Long URL
 *            |
 *            v
 *       +----------+
 *       |    fx    |
 *       +----+-----+
 *            |
 *            v
 *       Short Code
 *            |
 *            v
 *   https://tinyurl.com/qtj5opu
 *
 *         Short Code
 *            |
 *            v
 *       Lookup Mapping
 *            |
 *            v
 *         Long URL
 *            |
 *            v
 *         Redirect
 *
 * Yaani system ka main goal hai:
 * - "Long URL ko reliably ek unique short code se map karna aur future mein us 
 *    short code se original URL ko quickly retrieve kar pana."
 */

In [ ]:
/**
 * Data Model:
 * - Ab hum decide karenge ki Short URL aur Long URL ki mapping ko system mein kaise 
 *   store karenge.
 * - URL Shortener mein humein mainly ye mapping maintain karni hai:
 *
 *         Short URL
 *             |
 *             v
 *         Long URL
 *
 * - Example:
 *
 *         abc123
 *            |
 *            v
 *         https://www.amazon.com/product/123
 *
 *
 * 1. Why Not Store Everything in Memory?
 *    - Ek simple approach ho sakta hai ki hum saari URL mappings application server ki
 *      memory mein Hash Table ke form mein store karein.
 *
 *         Application Server
 *                |
 *                v
 *            Hash Table
 *                |
 *          +-----+-----+
 *          |           |
 *          v           v
 *       abc123       xyz789
 *          |           |
 *          v           v
 *       Long URL    Long URL
 *
 *    - Ye lookup ke liye bahut fast hai.
 *    - Lekin real-world large-scale system mein problem hai:
 *
 *         100M URLs/day
 *                |
 *                v
 *         Huge Number of Mappings
 *                |
 *                v
 *         Memory Requirement Grows
 *                |
 *                v
 *         RAM Limit Reached
 *
 *    - Server restart/crash hone par in-memory data lose bhi ho sakta hai.
 *    - Isliye production system mein sirf application memory par depend karna suitable
 *      nahi hai.
 *
 * 2. Relational Database:
 *    - Hum URL mappings ko persistent relational database mein store kar sakte hain.
 *
 *         URL Shortener
 *              |
 *              v
 *       +--------------+
 *       | Relational DB|
 *       +--------------+
 *              |
 *              v
 *          URL Table
 *
 *    - Database ka benefit:
 *       -> Data persistent rehta hai.
 *       -> Server restart hone par data lose nahi hota.
 *       -> Large amount of data store kiya ja sakta hai.
 *       -> Efficient lookups ke liye indexes use kar sakte hain.
 *       -> Future mein additional metadata store kar sakte hain.
 *
 *
 *    3. URL Table:
 *       - Basic table kuch is tarah ho sakti hai:
 *         +--------------------------------+
 *         |              url               |
 *         +--------+-----------+-----------+
 *         | PK     | shortURL  | longURL   |
 *         +--------+-----------+-----------+
 *         |   1    | abc123    | https://..|
 *         |   2    | xyz789    | https://..|
 *         |   3    | pqr456    | https://..|
 *         +--------+-----------+-----------+
 *
 *      - Columns:
 *        id:
 *           -> Primary Key.
 *           -> Har record ko uniquely identify karta hai.
 *        shortURL:
 *           -> Generated short code/URL.
 *           -> Iske through original Long URL lookup ki jaayegi.
 *        longURL:
 *           -> Original URL jo user ne provide kiya tha.
 *
 *
 *    Example:
 *    - id = 1
 *    - shortURL: abc123
 *    - longURL: https://www.amazon.com/product/123
 *    - Mapping:
 *       abc123
 *          |
 *          v
 *       https://www.amazon.com/product/123
 *
 *
 * 4. Metadata:
 *    - Real-world system mein table ko future requirements ke according extend bhi 
 *      kiya ja sakta hai.
 *      +------------------------------------------------+
 *      | id | shortURL | longURL | createdAt | clicks   |
 *      +------------------------------------------------+
 *      |  1 | abc123   | https.. | 2026-08.. | 15230    |
 *      +------------------------------------------------+
 *
 *    - Additional fields ho sakte hain:
 *       -> creation date
 *       -> click count
 *       -> owner/user ID
 *       -> expiration date
 *       -> custom alias
 *    - Yaani basic URL mapping ke saath system additional metadata bhi store kar sakta
 *      hai.
 *
 *
 * 5. Read Flow:
 *    - Jab user short URL open kare:
 *
 *         GET /abc123
 *                |
 *                v
 *         Extract shortURL
 *                |
 *                v
 *         Database Lookup
 *                |
 *                v
 *         Find longURL
 *                |
 *                v
 *         HTTP Redirect
 *
 *
 *    - Database lookup:
 *
 *         shortURL = "abc123"
 *                |
 *                v
 *         +----------------+
 *         |   URL Table    |
 *         +-------+--------+
 *                 |
 *                 v
 *         longURL = "https://..."
 *                 |
 *                 v
 *              Redirect
 *
 *
 * Simple words mein:
 * - Application memory fast hoti hai, lekin large-scale URL mappings ko permanently 
 *   store karne ke liye sufficient nahi hai.
 * - Isliye hum relational database use kar sakte hain jo:
 *   -> Persistence provide kare.
 *   -> Durability provide kare.
 *   -> Large-scale data store kare.
 *   -> Efficient indexed lookups support kare.
 *   -> Additional metadata store karne ki flexibility de.
 *
 *
 * Core Idea:
 *
 *           Client
 *              |
 *              v
 *         URL Shortener
 *              |
 *              v
 *         Relational DB
 *              |
 *              v
 *  +-------------------------+
 *  |         url             |
 *  +----+------+-------------+
 *  | id | shortURL | longURL |
 *  +----+------+-------------+
 *              |
 *              v
 *         Persistent Mapping
 *
 * - Yaani URL Shortener ka core data model simple hai: shortURL -> longURL
 * - Aur relational database is mapping ko reliably aur persistently store karne ka 
 *   ek straightforward choice hai.
 */

In [ ]:
/**
 * Hash Function:
 * - The hash function maps a long URL -> short hash value (hashValue) used in the
 *   short URL
 * - Example format: https://tinyurl.com/{hashValue}
 * - Hash Value Length Calculation:
 *   - Allowed characters: [0-9, a-z, A-Z] -> 62 characters
 *   - Goal: Support 365 billion URLs
 * - Length of hashValue = 7 characters
 *   - supports trillions of URLs, sufficient for 10-year growth.
 * 
 * +---+------------------------------------------+
 * | n | Maximum number of URLs                   |
 * +---+------------------------------------------+
 * | 1 | 62^1 = 62                                |
 * | 2 | 62^2 = 3,844                             |
 * | 3 | 62^3 = 238,328                           |
 * | 4 | 62^4 = 14,776,336                        |
 * | 5 | 62^5 = 916,132,832                       |
 * | 6 | 62^6 = 56,800,235,584                    |
 * | 7 | 62^7 = 3,521,614,606,208 = ~3.5 trillion |
 * | 8 | 62^8 = 218,340,105,584,896               |
 * +---+------------------------------------------+
*/

In [ ]:
/**
 * Hash Function:
 *
 *    Ab hum samjhenge ki Long URL ko Short URL mein convert karne
 *    ke liye hashValue kaise represent kiya ja sakta hai aur
 *    uski length kaise decide karenge.
 *
 *    Basic idea:
 *
 *         Long URL
 *            |
 *            v
 *       Hash Function
 *            |
 *            v
 *        hashValue
 *            |
 *            v
 *    https://tinyurl.com/{hashValue}
 *
 *    Example:
 *
 *         Long URL:
 *
 *         https://www.amazon.com/very/long/product/url
 *
 *                    |
 *                    v
 *
 *              Hash Function
 *                    |
 *                    v
 *
 *                 qtj5opu
 *
 *                    |
 *                    v
 *
 *         https://tinyurl.com/qtj5opu
 *
 *
 *    HashValue Length Calculation:
 *
 *    Ab question hai:
 *
 *         "HashValue kitne characters ka hona chahiye?"
 *
 *    Hamare system ko approximately:
 *
 *         365 billion URLs
 *
 *    support karne hain.
 *
 *    Isliye hashValue ke paas enough unique combinations hone chahiye
 *    taaki har URL ko ek unique short code diya ja sake.
 *
 *
 *    Allowed Characters:
 *
 *    Hum short URL mein sirf ye characters use karenge:
 *
 *         0-9  -> 10 characters
 *         a-z  -> 26 characters
 *         A-Z  -> 26 characters
 *
 *         Total = 10 + 26 + 26
 *               = 62 characters
 *
 *    Is character set ko Base62 kehte hain.
 *
 *
 *    Agar hashValue ki length "n" hai, to maximum possible
 *    combinations honge:
 *
 *         62^n
 *
 *
 *    Example:
 *
 *         1 character:
 *         62^1 = 62
 *
 *         2 characters:
 *         62^2 = 3,844
 *
 *         3 characters:
 *         62^3 = 238,328
 *
 *    Jaise-jaise hashValue ki length badhti hai,
 *    possible unique combinations exponentially increase hoti hain.
 *
 *
 *    +---+----------------------------------------------+
 *    | n | Maximum Number of URLs                       |
 *    +---+----------------------------------------------+
 *    | 1 | 62^1 = 62                                    |
 *    | 2 | 62^2 = 3,844                                 |
 *    | 3 | 62^3 = 238,328                               |
 *    | 4 | 62^4 = 14,776,336                            |
 *    | 5 | 62^5 = 916,132,832                           |
 *    | 6 | 62^6 = 56,800,235,584                        |
 *    | 7 | 62^7 = 3,521,614,606,208 ≈ 3.5 trillion      |
 *    | 8 | 62^8 = 218,340,105,584,896                   |
 *    +---+----------------------------------------------+
 *
 *
 *    7 Characters:
 *
 *         62^7
 *            |
 *            v
 *         3,521,614,606,208
 *            |
 *            v
 *         ≈ 3.5 trillion
 *
 *    Hamara requirement:
 *
 *         ≈ 365 billion URLs
 *
 *    Comparison:
 *
 *         Available:
 *         ~3.5 trillion
 *
 *         Required:
 *         ~365 billion
 *
 *         Therefore:
 *
 *         ~3.5 trillion > ~365 billion
 *
 *    Isliye 7-character hashValue hamare estimated
 *    10-year growth ke liye sufficient hai.
 *
 *
 *    Why 7 Characters?
 *
 *         Shorter Code
 *              |
 *              v
 *         Fewer Combinations
 *
 *         Longer Code
 *              |
 *              v
 *         More Combinations
 *
 *    Humein aisa code chahiye jo:
 *
 *         -> As short as possible ho.
 *         -> Required number of unique URLs support kare.
 *
 *    7 characters isliye choose kiye ja sakte hain kyunki:
 *
 *         7 characters
 *              |
 *              v
 *         ~3.5 trillion combinations
 *              |
 *              v
 *         > 365 billion required URLs
 *
 *
 *    Important:
 *
 *    Sirf 62^7 combinations available hona ye guarantee nahi karta
 *    ki koi bhi arbitrary hash function collision-free hoga.
 *
 *    Example:
 *
 *         Long URL A ---> abc1234
 *         Long URL B ---> abc1234
 *
 *                         X
 *                     Collision
 *
 *    Isliye actual system design mein hash generation ke saath
 *    collision handling bhi required hogi.
 *
 *
 *    Simple words mein:
 *
 *    Hum Base62 ke 62 possible characters use karte hain.
 *
 *    Agar hashValue 7 characters ka hai:
 *
 *         62^7 ≈ 3.5 trillion
 *
 *    unique short codes possible hain.
 *
 *    Hamara estimated requirement:
 *
 *         ≈ 365 billion URLs
 *
 *    Isliye capacity ke point of view se 7-character hashValue
 *    sufficient hai.
 *
 *
 *    Core Idea:
 *
 *         Long URL
 *            |
 *            v
 *       Hash Function
 *            |
 *            v
 *       7 Character Code
 *            |
 *            v
 *    https://tinyurl.com/qtj5opu
 *
 *         Base62:
 *
 *         0-9 + a-z + A-Z
 *                |
 *                v
 *             62 chars
 *                |
 *                v
 *              62^7
 *                |
 *                v
 *          ~3.5 trillion
 *
 *         Required:
 *
 *          ~365 billion
 *
 *         Therefore:
 *
 *         7 characters -> sufficient capacity
 *                         for our 10-year estimate.
 */

In [ ]:
/**
 * Hash Function Strategies:
 * 1. Hash + Collision Resolution
 *    - Hash long URL -> fixed-length value
 *    - Handle collisions via retries or chaining
 * 2. Base62 Conversion
 *    - Convert unique numeric ID -> Base62 string
 *    - Ensures unique, short, and predictable hashValue
*/

In [ ]:
/**
 * Hash + Collision Resolution:
 *
 *    Ab hum first strategy ko detail mein samjhenge:
 *
 *         "Hash + Collision Resolution"
 *
 *    Hamara goal hai:
 *
 *         Long URL
 *             |
 *             v
 *         Hash Function
 *             |
 *             v
 *       7-character hashValue
 *             |
 *             v
 *       https://tinyurl.com/{hashValue}
 *
 *    Yaani har Long URL ko ek 7-character short code mein convert
 *    karna hai.
 *
 *
 *    Step-1: Apply Standard Hash Function
 *
 *    Sabse pehle Long URL par ek standard hash function apply
 *    karte hain.
 *
 *         Long URL
 *             |
 *             v
 *        Hash Function
 *             |
 *             v
 *        Hash Value
 *
 *    Common hash functions:
 *
 *         -> CRC32
 *         -> MD5
 *         -> SHA-1
 *
 *
 *    Example Long URL:
 *
 *         https://en.wikipedia.org/wiki/Systems-design
 *
 *                     |
 *                     v
 *               Hash Function
 *                     |
 *                     v
 *                Hash Value
 *
 *
 *    Problem:
 *
 *    Humein sirf 7 characters ka short code chahiye,
 *    lekin standard hash functions ka output usually isse
 *    kaafi bada hota hai.
 *
 *    Example:
 *
 *    +---------------+--------------------------------------+
 *    | Hash Function | Hash Value                           |
 *    +---------------+--------------------------------------+
 *    | CRC32         | 5cb54054                             |
 *    | MD5           | 5a62jkhasbjfweio79868stigdufbj       |
 *    | SHA-1         | oinwefioiwerg8ih9wer89wgug9ws8eegh9w8eg |
 *    +---------------+--------------------------------------+
 *
 *    CRC32 ka output bhi hexadecimal form mein 8 characters
 *    ka hota hai.
 *
 *         CRC32
 *           |
 *           v
 *       5cb54054
 *           |
 *           v
 *       8 characters
 *
 *    Lekin hamari requirement:
 *
 *         7 characters
 *
 *    Isliye humein hash ko truncate karna padega.
 *
 *
 *    Step-2: Truncate Hash
 *
 *    Example:
 *
 *         Full Hash:
 *
 *         5cb54054
 *
 *         First 7 characters:
 *
 *         5cb5405
 *
 *         Short URL:
 *
 *         https://tinyurl.com/5cb5405
 *
 *
 *    Lekin yahan ek important problem aa jaati hai:
 *
 *         Collision
 *
 *
 *    Collision ka matlab:
 *
 *    Do different Long URLs same 7-character hashValue
 *    generate kar dein.
 *
 *
 *         Long URL A
 *              |
 *              v
 *          Hash Function
 *              |
 *              v
 *           abc1234
 *
 *         Long URL B
 *              |
 *              v
 *          Hash Function
 *              |
 *              v
 *           abc1234
 *
 *              X
 *          Collision
 *
 *    Ab dono URLs ka same Short URL ban gaya:
 *
 *         https://tinyurl.com/abc1234
 *
 *    System ko samajh nahi aayega ki ye Short URL
 *    Long URL A ko point kare ya Long URL B ko.
 *
 *
 *    Why Collision is Possible?
 *
 *    Hamare paas 7-character Base62 code ke approximately:
 *
 *         62^7 ≈ 3.5 trillion
 *
 *    possible combinations hain.
 *
 *    Lekin agar hum kisi standard hash ka output truncate
 *    kar rahe hain, to different Long URLs same truncated
 *    value produce kar sakte hain.
 *
 *         Full Hash A -> ...abc1234
 *         Full Hash B -> ...abc1234
 *
 *                         |
 *                         v
 *                   Same 7 chars
 *                         |
 *                         v
 *                     Collision
 *
 *
 *    Step-3: Collision Resolution
 *
 *    Isliye Short URL generate karne ke baad database mein
 *    check karna padega ki generated hashValue already exist
 *    karta hai ya nahi.
 *
 *
 *         Generate hashValue
 *                |
 *                v
 *          Database Lookup
 *                |
 *          +-----+-----+
 *          |           |
 *        Exists     Not Exists
 *          |           |
 *          v           v
 *       Collision    Store
 *          |
 *          v
 *       Generate
 *       New Hash
 *
 *
 *    Agar hashValue already exist karta hai, to collision
 *    resolve karna padega.
 *
 *
 *    Approach: Recursive Appending
 *
 *    Collision hone par original Long URL ke saath ek
 *    predefined string append kar sakte hain.
 *
 *    Example:
 *
 *         Original URL:
 *
 *         https://example.com/page
 *
 *                     |
 *                     v
 *                Hash Function
 *                     |
 *                     v
 *                  abc1234
 *
 *                     |
 *                     v
 *                 Collision
 *
 *
 *    Ab URL ke end mein predefined string append karte hain:
 *
 *         https://example.com/page + "1"
 *
 *                     |
 *                     v
 *                Hash Function
 *                     |
 *                     v
 *                  x7k92Lm
 *
 *                     |
 *                     v
 *               Check Database
 *
 *
 *    Agar phir bhi collision ho:
 *
 *         https://example.com/page + "2"
 *
 *                     |
 *                     v
 *                Hash Function
 *                     |
 *                     v
 *                  p9Qa21X
 *
 *    Ye process tab tak repeat hota hai jab tak unique
 *    7-character hashValue nahi mil jaata.
 *
 *
 *    Complete Flow:
 *
 *         Long URL
 *            |
 *            v
 *       Hash Function
 *            |
 *            v
 *       Truncate to 7 chars
 *            |
 *            v
 *       Check Database
 *            |
 *        +---+---+
 *        |       |
 *      Unique  Collision
 *        |       |
 *        v       v
 *      Store   Append String
 *                |
 *                v
 *           Recompute Hash
 *                |
 *                v
 *           Check Again
 *                |
 *                v
 *             Repeat
 *
 *
 *    Example:
 *
 *         URL
 *          |
 *          v
 *       Hash -> abc1234
 *          |
 *          v
 *       Already Exists?
 *          |
 *         YES
 *          |
 *          v
 *       Append "1"
 *          |
 *          v
 *       Hash -> x7k92Lm
 *          |
 *          v
 *       Already Exists?
 *          |
 *          NO
 *          |
 *          v
 *       Store Mapping
 *
 *
 *    Drawback:
 *
 *    Is approach ka main drawback hai:
 *
 *         Database Lookups
 *
 *    Har generated hashValue ke liye uniqueness check karna
 *    padta hai.
 *
 *         Generate Hash
 *              |
 *              v
 *         DB Lookup
 *              |
 *          Collision?
 *              |
 *              v
 *         Generate Again
 *              |
 *              v
 *         DB Lookup Again
 *
 *    Agar collisions frequent ho gaye, to ek single URL create
 *    karne ke liye multiple database lookups aur hash calculations
 *    perform karne pad sakte hain.
 *
 *
 *         More Collisions
 *                |
 *                v
 *         More Hash Attempts
 *                |
 *                v
 *         More DB Lookups
 *                |
 *                v
 *         Higher Write Latency
 *
 *
 *    Simple words mein:
 *
 *    Hash + Collision Resolution approach mein:
 *
 *         1. Long URL ko hash karo.
 *
 *         2. Hash ko 7 characters tak truncate karo.
 *
 *         3. Database mein check karo ki ye short code already
 *            use ho raha hai ya nahi.
 *
 *         4. Agar unique hai:
 *               -> Store mapping.
 *
 *         5. Agar collision hai:
 *               -> URL mein predefined string append karo.
 *               -> Dobara hash karo.
 *               -> Dobara uniqueness check karo.
 *
 *         6. Ye process unique hashValue milne tak repeat hota hai.
 *
 *
 *    Core Idea:
 *
 *         Long URL
 *             |
 *             v
 *        Standard Hash
 *             |
 *             v
 *        7-char Hash
 *             |
 *             v
 *       +-----+-----+
 *       |           |
 *     Unique     Collision
 *       |           |
 *       v           v
 *      Store    Append String
 *                   |
 *                   v
 *              Rehash
 *                   |
 *                   v
 *             Check Again
 *
 *    Main Trade-off:
 *
 *         Simple Hashing
 *              +
 *         Collision Resolution
 *              |
 *              v
 *         But potentially expensive
 *         DB lookups on collisions.
 */

In [ ]:
/**
 * Base62 Conversion:
 * - Goal: Convert a unique
*/